# FraudShield v3 — Train Only (Same Drive, New Colab)

## Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. When Drive mounts → sign in with your **SAME** Google account (the one with the data)

## What this does:
- Loads `train_v3.csv` and `val_v3.csv` from your existing Drive
- Trains MuRIL v3 with class weights + improved LR (~60-90 min on T4)
- Evaluates, tests your real Indian SMS, saves model back to Drive

## What's new vs v2:
- **EPOCHS 5** (was 4) — early stopping prevents overfit
- **LR 1e-5** (was 2e-5) — MuRIL is sensitive to learning rate
- **WeightedTrainer** — penalises missing fraud more than false alarms
- **12 data sources** combined locally (OTP/KYC/UPI/Courier/Job/Hinglish fraud)

## Your Drive paths (same as v2 — just upload 3 new files):
```
My Drive/fraudshield/model/data/cleaned/train_v3.csv        <- upload this
My Drive/fraudshield/model/data/cleaned/val_v3.csv          <- upload this
My Drive/fraudshield/model/data/cleaned/test_v3_LOCKED.csv  <- upload this
My Drive/fraudshield/model/models/muril-fraud-v3/           <- saves here
```

## Generate the 3 files locally first:
```bash
python ml/scripts/generate/generate_synthetic_sms_v2.py
python ml/scripts/process/prepare_datasets_v3.py
```
Then upload `ml/data/processed/train_v3.csv`, `val_v3.csv`, `test_v3_LOCKED.csv` to the same Drive folder.

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install transformers datasets accelerate scikit-learn pandas numpy torch --quiet
print('Done')

## Step 2 — Mount Google Drive
Sign in with your **SAME** Google account when the popup appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Same paths as v2 ──────────────────────────────────────────
CLEANED_DIR = '/content/drive/MyDrive/fraudshield/model/data/cleaned'
MODELS_DIR  = '/content/drive/MyDrive/fraudshield/model/models'
REGISTRY    = f'{MODELS_DIR}/muril-fraud-v3'

os.makedirs(REGISTRY, exist_ok=True)

# Verify all files exist
print('Checking files...')
all_ok = True
for f in ['train_v3.csv', 'val_v3.csv', 'test_v3_LOCKED.csv']:
    p = f'{CLEANED_DIR}/{f}'
    ok = os.path.exists(p)
    if not ok:
        all_ok = False
    size = round(os.path.getsize(p) / 1024 / 1024, 1) if ok else 0
    print(f'  {"OK" if ok else "MISSING"} {f}  ({size} MB)')

print()
if all_ok:
    print('All files found. Ready to train.')
else:
    print('ERROR: Some files missing. Check your Drive path.')
    print(f'  Expected at: {CLEANED_DIR}')

## Step 3 — Load CSVs

In [ ]:
import pandas as pd

train = pd.read_csv(f'{CLEANED_DIR}/train_v3.csv', encoding='latin-1')
val   = pd.read_csv(f'{CLEANED_DIR}/val_v3.csv',   encoding='latin-1')

# Clean
for df in [train, val]:
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
train = train.dropna(subset=['text','label'])
val   = val.dropna(subset=['text','label'])
train = train[train['label'].isin([0,1])].copy()
val   = val[val['label'].isin([0,1])].copy()
train['label'] = train['label'].astype(int)
val['label']   = val['label'].astype(int)

print(f'Train : {len(train):,} rows  (legit={(train.label==0).sum():,}, fraud={train.label.sum():,})')
print(f'Val   : {len(val):,} rows  (legit={(val.label==0).sum():,}, fraud={val.label.sum():,})')
print()
print('Sample legit:')
print(' ', train[train.label==0].iloc[0]['text'])
print('Sample fraud:')
print(' ', train[train.label==1].iloc[0]['text'])

## Step 4 — Tokenize
Downloads MuRIL tokenizer and tokenizes dataset. Takes ~3-5 min.

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'google/muril-base-cased'
MAX_LEN    = 256

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY — switch to T4 GPU in Runtime settings"}')
print()
print('Downloading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Done')

train_ds = Dataset.from_pandas(train[['text','label']].astype({'label': int}))
val_ds   = Dataset.from_pandas(val[['text','label']].astype({'label': int}))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True,
                     max_length=MAX_LEN, padding='max_length')

print('Tokenizing train...')
train_ds = train_ds.map(tokenize, batched=True, batch_size=512)
print('Tokenizing val...')
val_ds   = val_ds.map(  tokenize, batched=True, batch_size=512)

train_ds = train_ds.rename_column('label', 'labels')
val_ds   = val_ds.rename_column(  'label', 'labels')
train_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
val_ds.set_format(  'torch', columns=['input_ids','attention_mask','labels'])

print('Tokenization complete')

## Step 5 — Train MuRIL v3
~60–90 min on T4 GPU. Checkpoints saved to Drive every epoch so progress is safe.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

BATCH_SIZE = 16
EPOCHS     = 5       # was 4 — early stopping prevents overfit
LR         = 1e-5    # was 2e-5 — MuRIL is sensitive to LR

# Checkpoints saved directly to Drive — safe even if Colab disconnects
CKPT_DIR = f'{MODELS_DIR}/checkpoints_v3'
os.makedirs(CKPT_DIR, exist_ok=True)

# Class weights — penalises missing fraud more than false alarms
fraud_n = int(train['label'].sum())
legit_n = len(train) - fraud_n
ratio   = legit_n / fraud_n
print(f'Class ratio legit/fraud = {ratio:.2f}  →  weight=[1.0, {ratio:.2f}]')

weight = torch.tensor([1.0, ratio], dtype=torch.float)
if torch.cuda.is_available():
    weight = weight.cuda()

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print('Loading MuRIL model...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print('Model loaded')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy' : round(accuracy_score( labels, preds), 4),
        'precision': round(precision_score(labels, preds, zero_division=0), 4),
        'recall'   : round(recall_score(   labels, preds, zero_division=0), 4),
        'f1'       : round(f1_score(       labels, preds, zero_division=0), 4),
    }

args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',        # saves checkpoint to Drive each epoch
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    save_total_limit            = 2,
)

trainer = WeightedTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print()
print('Training started...')
print('Checkpoints saved to Drive after every epoch')
print('Estimated: ~60-90 min on T4 GPU')
print()
trainer.train()

## Step 6 — Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_out = trainer.predict(val_ds)
preds    = np.argmax(pred_out.predictions, axis=1)
labels   = val_ds['labels']

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print('Validation Results (v3)')
print(f'  Accuracy  : {accuracy_score(labels, preds):.4f}')
print(f'  Precision : {precision_score(labels, preds):.4f}')
print(f'  Recall    : {recall_score(labels, preds):.4f}')
print(f'  F1        : {f1_score(labels, preds):.4f}')
print(f'  FP rate   : {fpr:.4f}  (target < 0.02)')
print()
print(classification_report(labels, preds, target_names=['Legit','Fraud']))

## Step 7 — Test your real SMS

In [ ]:
test_messages = [
    # LEGIT — must NOT be flagged as fraud
    ('Dear BOB UPI User: Your account is credited with INR 145.00 on 2026-03-23 07:53:11 PM by UPI Ref No 195310192638; AvlBal: Rs641.94 - BOB',
     0, 'BOB credit — MUST be LEGIT'),
    ('Rs.500.00 Dr. from A/C XXXXXX5608 and Cr. to nptushyent@okaxis. Ref:644933179447. AvlBal:Rs121.94(2026:03:24 05:54:28). Not you? Call 18005700/5000-BOB',
     0, 'BOB debit @okaxis — MUST be LEGIT'),
    ('Rs.130.00 Dr. from A/C XXXXXX5608 and Cr. to nptushyent@okicici. Ref:530359835274. AvlBal:Rs2054.30(2025:10:30 12:52:41). Not you? Call 18005700/5000-BOB',
     0, 'BOB debit @okicici — MUST be LEGIT'),
    ('AXISBK: INR 2500.00 credited to A/c XX4139 on 15-01-26. Avl Bal INR 45,200.00',
     0, 'Axis credit — MUST be LEGIT'),
    ('HDFCBK: OTP for HDFC Net Banking login is 847293. Valid for 10 mins. Do NOT share with anyone.',
     0, 'OTP SMS — MUST be LEGIT'),
    ('Dear customer, your Airtel bill of Rs.399 is due on 31-Mar. Pay at airtel.in',
     0, 'Airtel bill — MUST be LEGIT'),
    ('SBIBNK: INR 1500 debited from A/c XX4521 to VPA xyz@upi on 27-Mar-26. Avl Bal Rs.8420.15',
     0, 'SBI UPI debit — MUST be LEGIT'),
    # FRAUD — must be caught
    ('SBI-KYC: Your account will be blocked. Verify at sbi-kyc.net immediately or your account will be permanently closed.',
     1, 'KYC scam — MUST be FRAUD'),
    ('Congratulations! You have won Rs.500000 in RBI Lucky Draw. Claim at rbi-lucky.net within 24hrs.',
     1, 'Lottery scam — MUST be FRAUD'),
    ('Work from home opportunity! Earn Rs.45000/month. Pay Rs.499 registration fee at gpay-alert.co/jobs.',
     1, 'Job scam — MUST be FRAUD'),
    ('Your SBI OTP is 847291. Share with our agent to unblock your account immediately. -SBI-ALERT',
     1, 'OTP share scam — MUST be FRAUD'),
    ('Your parcel is held at customs. Pay Rs.49 at fedex-india.net to release. FEDEX India.',
     1, 'Courier scam — MUST be FRAUD'),
    ('Aapka SBI account band ho jayega. Abhi verify karein: sbi-verify.in -SBI-KYC',
     1, 'Hindi KYC scam — MUST be FRAUD'),
    ('You received Rs.5000 UPI reward from HDFC. Click upi-reward.co to claim. Expires 1hr.',
     1, 'UPI reward scam — MUST be FRAUD'),
]

model.eval()
correct = 0
print('Real SMS Test\n')
for text, true_label, description in test_messages:
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_LEN, padding='max_length'
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred      = torch.argmax(logits, dim=1).item()
    conf      = torch.softmax(logits, dim=1).max().item()
    status    = 'PASS' if pred == true_label else 'FAIL'
    label_str = 'FRAUD' if pred == 1 else 'LEGIT'
    if pred == true_label:
        correct += 1
    print(f'[{status}] [{label_str}] {conf:.1%}  {description}')
    print(f'       "{text[:90]}"')
    print()

print(f'Score: {correct}/{len(test_messages)} correct')
if correct == len(test_messages):
    print('All passing — safe to save')
else:
    print('Some failures — check dataset or run more epochs')

## Step 8 — Save model to Drive
Saves to `My Drive/fraudshield/model/models/muril-fraud-v3`

In [ ]:
import json

model.save_pretrained(REGISTRY)
tokenizer.save_pretrained(REGISTRY)

results = {
    'model_version'       : 'muril-fraud-v3',
    'base_model'          : MODEL_NAME,
    'train_size'          : len(train),
    'val_size'            : len(val),
    'accuracy'            : round(accuracy_score(labels, preds), 4),
    'precision'           : round(precision_score(labels, preds), 4),
    'recall'              : round(recall_score(labels, preds), 4),
    'f1'                  : round(f1_score(labels, preds), 4),
    'false_positive_rate' : round(fpr, 4),
    'epochs_config'       : EPOCHS,
    'learning_rate'       : LR,
    'class_weight_ratio'  : round(ratio, 3),
    'sms_test_score'      : f'{correct}/{len(test_messages)}',
}

with open(f'{REGISTRY}/val_results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Model saved to:')
print(f'  {REGISTRY}')
print()
print('Results:')
for k, v in results.items():
    print(f'  {k:<25}: {v}')
print()
print('Files saved:')
for f in sorted(os.listdir(REGISTRY)):
    size_mb = os.path.getsize(f'{REGISTRY}/{f}') / 1024 / 1024
    print(f'  {f:<45} {size_mb:.1f} MB')
print()
print('Done. Next: update FastAPI to load muril-fraud-v3')